# RL training for LLM agents — 1. GRPO from scratch

### Building an RL trainer from nothing but PyTorch

Frameworks like [verl](https://github.com/volcengine/verl) make it possible to train LLMs with reinforcement learning (RL) across many GPUs, but they are also large, layered codebases: Ray workers, FSDP, an inference engine with weight sync, agent loops, a few hundred config options. When something goes wrong, or when we want to train an agent that doesn't fit the framework's mould, it is hard to know which of those layers matters, and why it exists at all.

Jeremy Howard's [*What is `torch.nn` really?*](https://pytorch.org/tutorials/beginner/nn_tutorial.html) has a great answer to this kind of problem: train a model with nothing but tensors, then replace one hand-written piece at a time with its library equivalent, until only idiomatic library code is left. After every swap the code still runs, and we know exactly what each library piece does, because we wrote it first. This series does the same for RL training of LLMs:

| # | Notebook | What it builds | Sections |
|---|---|---|---|
| 1 | [GRPO from scratch](1_grpo_from_scratch.ipynb) | an RL trainer in plain PyTorch, then a real inference engine (vLLM) and weight sync | §0–§2 |
| 2 | [From our trainer to verl](2_from_trainer_to_verl.ipynb) | the same trainer, rebuilt from verl's pieces: `DataProto`, `core_algos`, Ray actors, worker groups; then plain verl | §3–§6 |
| 3 | [Training real agents](3_training_real_agents.ipynb) | multi-turn tool use, recording a real agent at the HTTP boundary, Agent Lightning, and verl + Agent Lightning end to end | §7–§10 |
| 4 | [Rewards: exact checks and LLM judges](4_rewards_and_judges.ipynb) | what a reward measures, and an LLM judge next to an exact check | own numbering |
| 5 | [Calc-X end to end](5_calc_x_end_to_end.ipynb) | one training step driven by hand through Agent Lightning, then a real training run | own numbering |

Notebooks 1–3 share one set of section numbers, so a reference like "§1.5" or "§8.1" always points to the same place, whichever notebook it appears in. Each notebook runs on its own: code from earlier notebooks is imported from `tutorial/lib/`, where it is copied verbatim.

**In this notebook** we build the trainer itself, and then make its slowest part fast:

| § | We write | Then replace it with | The problem that forces the swap |
|---|---|---|---|
| 1 | GRPO in plain PyTorch: sample, score, log-probs, advantages, PPO-clip loss | | |
| 2 | | `model.generate` → **vLLM** | generation is most of the step time; now we need **weight sync** |

**What we need to know already.** PyTorch, and roughly how an LLM is fine-tuned. Everything RL-specific (policies, log-probs, rewards, advantages, PPO) is introduced when we first need it.

**Hardware and time.** One GPU with about 40 GB of free memory, and a few minutes on an H100 (plus a 1 GB model download). The policy is `Qwen2.5-0.5B-Instruct`, small enough to train in minutes.

## 0. Setup

Paths, the model name and the random seed live in a small shared module, `tutorial/lib/common.py`, so that every notebook in the series uses the same ones. `setup()` creates the working folders (logs and data go to `/tmp/rl-from-scratch/`; weights are handed from trainer to inference engine through `/dev/shm/rl-from-scratch/`, which lives in RAM, not on disk), seeds the random number generators, and checks that the GPU is free.

In [1]:
import gc
import json
import os
import random
import re
import socket
import subprocess
import sys
import threading
import time
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "calc_x").is_dir())
sys.path.insert(0, str(REPO / "tutorial/lib"))  # the series' shared code
from common import (CALC_X_DATA, CALC_X_MODEL, DEVICE, GENERATED, MODEL, SEED, SERVICES, SHM, VENV_BIN, WORK,  # noqa: E402
                    follow_log, gpu_free_gib, log_tail, setup, start_service, stop_service, wait_until_healthy)

setup()

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402

repo: /home/azureuser/localfiles/agl-rl-tutorials
python: /home/azureuser/localfiles/agl-rl-tutorials/.venv/bin/python
GPU: NVIDIA H100 NVL, 93 of 93 GiB free


## 1. GRPO in plain PyTorch

In this section we build a complete RL trainer for an LLM out of nothing but PyTorch and a Hugging Face model, small enough to read in one sitting. Every later section replaces one piece of it with the library version.

### Why RL is different from supervised fine-tuning

In **supervised fine-tuning (SFT)**, every training example comes with the correct answer, written by someone else. The loss compares the model's prediction with that answer token by token, so every token is told exactly what it should have been: "you wrote `20` here, the answer has `14`". When the model makes a mistake, we know *where* it went wrong and *what* it should have said instead.

In **RL** there is no correct answer to compare against. The model writes its own attempt, and all we get back is a single score for the whole thing: here, 1 if the final answer is right and 0 if it's wrong. The score doesn't say which token caused the mistake, or what the model should have written instead.

So RL uses the only lever left: **comparison**. We let the model attempt the same question several times. If some attempts score better than others, the better ones probably contain something worth doing more of, and the worse ones something worth doing less of. Training then nudges the weights so that every token of a better-than-average attempt becomes a little more likely, and every token of a worse-than-average attempt a little less likely. Over many questions, the tokens that really caused success get pushed up consistently, and the rest averages out.

| | SFT | RL |
|---|---|---|
| What we need | prompts **and** correct answers | prompts and a way to **check** an answer |
| Whose text we train on | someone else's (demonstrations) | the model's own (samples) |
| What each token is told | the token it should have been | nothing individually: one score for the whole response |
| Direction of the update | always "make this text more likely" | up for better-than-expected attempts, down for worse |

### The loop

We repeat these four steps many thousands of times:

```
          ┌───────────────────────── policy π_θ  (the LLM weights θ) ──────────────────────┐
          │                                                                                      │
question ─┼─► SAMPLE: n attempts per question ──► REWARD: is each answer correct?                │
          │                                                     │                                │
          │      ADVANTAGE: was this attempt better or worse than its siblings?                  │
          │                                                     │                                │
          └──◄── UPDATE: raise log π(tokens) when advantage > 0, lower it when advantage < 0 ◄───┘
```

1. **Sample** (§1.2): the model writes several attempts at each question.
2. **Reward** (§1.1): a function scores each attempt.
3. **Advantage** (§1.5): each attempt's score is compared with its siblings' (the other attempts at the same question), giving a number that is positive for better-than-average attempts and negative for worse.
4. **Update** (§1.5, §1.6): the weights move so that tokens of positive-advantage attempts become more likely and tokens of negative-advantage attempts less likely.

The algorithm we'll build is **GRPO** (Group Relative Policy Optimization), used for most recent reasoning models. The "group" is the set of attempts at one question, and "relative" means each attempt is judged against the rest of its group, exactly as in step 3.

### 1.1 The task

We need a task where a 0.5B model is **sometimes right and sometimes wrong**. That matters more than it sounds. RL learns from the *differences* between attempts at the same question (step 3 above). If the model always got a question right, all its attempts would score 1, none would be better than another, and there would be nothing to learn from. The same goes if it always got it wrong. We'll put numbers on this in §1.5.

Three-term integer arithmetic sits in the sweet spot: questions like "What is 19 * 99 + 10?", where the answer is a single integer that's easy to check.

The task has three pieces: a safe expression evaluator (to compute the right answers), a problem generator, and a reward function. Let's build each one, then bundle them into a module that later stages can import.

**`safe_eval`: evaluating expressions without `eval`.** To generate problems we need to compute answers like `19 * 99 + 10`, and in §7 the model will send us expressions to compute as a calculator tool. Python's built-in `eval` would do the arithmetic, but it runs *any* Python: if the model wrote `__import__('os').system('rm -rf ~')` as its "expression", `eval` would happily run it. Text written by a model must never reach `eval`.

Instead, we ask Python to **parse** the expression into a syntax tree without running it, then walk that tree ourselves, allowing only numbers and `+ - * /`. Here is the tree for `2 + 3 * 4`:

In [2]:
import ast

tree = ast.parse("2 + 3 * 4", mode="eval")  # parse only: nothing is executed
print(ast.dump(tree, indent=2))

Expression(
  body=BinOp(
    left=Constant(value=2),
    op=Add(),
    right=BinOp(
      left=Constant(value=3),
      op=Mult(),
      right=Constant(value=4))))


Reading the tree from the top: the whole expression is an addition (`BinOp` with `op=Add()`) whose left side is the number `2` and whose right side is *another* operation, `3 * 4`. Operator precedence is already settled by the shape of the tree: `*` sits lower down, so it gets computed first.

`safe_eval` below walks this tree with a small recursive function, `evaluate_node`. Each kind of node has one rule, and each rule asks for its children's values first:

```
evaluate_node(Expression)              → evaluate its body
  evaluate_node(BinOp +)               → evaluate left, evaluate right, then add
    evaluate_node(Constant 2)          → 2
    evaluate_node(BinOp *)             → evaluate left, evaluate right, then multiply
      evaluate_node(Constant 3)        → 3
      evaluate_node(Constant 4)        → 4
                                       → 3 * 4 = 12
                                       → 2 + 12 = 14
```

Anything without a rule (a function call like `__import__(...)`, a variable name, the power operator `**`) falls through to the final `raise`, so it is rejected before anything runs.

In [ ]:
import ast
import operator

_BINARY_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
_UNARY_OPS = {ast.USub: operator.neg, ast.UAdd: operator.pos}


def safe_eval(expression: str) -> float:
    """Evaluate `+ - * /` arithmetic by walking the syntax tree; anything else is rejected."""
    if len(expression) > 200:
        raise ValueError("expression too long")

    def evaluate_node(node):
        if isinstance(node, ast.Expression):  # the root of the tree
            return evaluate_node(node.body)
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):  # a number
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _BINARY_OPS:  # e.g. 2 + 3
            apply_op = _BINARY_OPS[type(node.op)]
            return apply_op(evaluate_node(node.left), evaluate_node(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:  # e.g. -5
            apply_op = _UNARY_OPS[type(node.op)]
            return apply_op(evaluate_node(node.operand))
        raise ValueError(f"unsupported expression: {expression!r}")  # names, calls, **, ...

    return evaluate_node(ast.parse(expression, mode="eval"))bb

Let's try it. Normal arithmetic should work; anything dangerous should be rejected.

In [4]:
print(safe_eval("2 + 3 * 4"))      # 14, not 20: standard precedence
print(safe_eval("100 - 37 + 5"))   # 68

for bad in ["__import__('os').system('id')", "2 ** 999999"]:
    try:
        safe_eval(bad)
    except ValueError as exc:
        print("rejected:", exc)

14
68
rejected: unsupported expression: "__import__('os').system('id')"
rejected: unsupported expression: '2 ** 999999'


**`make_problems`: generating questions.** Each problem is a dict with a `question` string and the integer `answer`, computed with `safe_eval`. Easy mode builds three-term expressions from small numbers; hard mode (used in §7) builds three-digit multiplications that a small model can't do reliably in its head, so it will need a calculator tool.

The `seed` makes generation repeatable: the same seed always gives the same problems, so runs are comparable. Training and evaluation use *different* seeds, so we always evaluate on questions the model never trained on.

In [5]:
import random


def make_problems(num_problems: int, seed: int, hard: bool = False) -> list[dict]:
    rng = random.Random(seed)
    problems = []
    for _ in range(num_problems):
        if hard:  # e.g. "482 * 917 - 3051": too big to multiply reliably without a calculator
            expression = " ".join([str(rng.randint(100, 999)), "*", str(rng.randint(100, 999)), "-", str(rng.randint(1000, 9999))])
        else:     # e.g. "37 + 12 * 5"
            expression = " ".join([
                str(rng.randint(2, 99)), rng.choice("+-*"),
                str(rng.randint(2, 99)), rng.choice("+-*"),
                str(rng.randint(2, 30)),
            ])
        problems.append({"question": f"What is {expression}?", "answer": int(safe_eval(expression))})
    return problems


problems = make_problems(6, seed=1)
for problem in problems:
    print(problem)

{'question': 'What is 19 * 99 + 10?', 'answer': 1891}
{'question': 'What is 17 - 99 - 17?', 'answer': -99}
{'question': 'What is 85 - 28 + 17?', 'answer': 74}
{'question': 'What is 5 - 57 * 26?', 'answer': -1477}
{'question': 'What is 2 * 59 - 25?', 'answer': 93}
{'question': 'What is 31 * 15 - 2?', 'answer': 463}


**`reward_fn`: scoring an attempt.** The reward is 1 if the last `\boxed{...}` in the response holds the right integer, and 0 otherwise. The pattern that finds the boxed number reads like this:

| Piece of the pattern | Matches |
|---|---|
| `\\boxed\{` | the literal text `\boxed{` |
| `\s*` | optional spaces |
| `(-?\d[\d,]*)` | the number we capture: an optional minus sign, a digit, then more digits or commas (so `1,234` works) |
| `\s*\}` | optional spaces and the closing `}` |

Two choices are deliberate:

* We take the **last** boxed number, because the model sometimes boxes an intermediate result before its final answer.
* The check is **strict**: the right number *without* the box scores 0. That forces the model to learn the output format as well as the arithmetic, and as we'll see in §2, learning the format is most of what early training does.

In [6]:
import re

BOXED = re.compile(r"\\boxed\{\s*(-?\d[\d,]*)\s*\}")


def reward_fn(text: str, answer: int) -> float:
    """1.0 if the last \\boxed{...} in `text` holds `answer`, else 0.0."""
    boxed_numbers = BOXED.findall(text)  # e.g. ["12", "1,234"]
    if not boxed_numbers:
        return 0.0
    final_answer = int(boxed_numbers[-1].replace(",", ""))
    return 1.0 if final_answer == answer else 0.0

Let's test a few cases. Pay attention to the third one — it gets the arithmetic right but scores 0.

In [7]:
for text, answer in [
    (r"... so \boxed{42}",  42),   # correct answer, correct format
    (r"\boxed{41}",         42),   # wrong answer
    ("The answer is 42.",   42),   # right answer, wrong format!
    (r"\boxed{1,234}",    1234),   # commas are fine
]:
    print(f"  reward = {reward_fn(text, answer)}  ←  {text!r}")

  reward = 1.0  ←  '... so \\boxed{42}'
  reward = 0.0  ←  '\\boxed{41}'
  reward = 0.0  ←  'The answer is 42.'
  reward = 1.0  ←  '\\boxed{1,234}'


`"The answer is 42."` scores 0, even though the arithmetic is right. That's worth dwelling on, because the reward is the *only* training signal we have, and rewards like this one are:

* **sparse**: one number at the very end of a response (and, later, of a multi-turn episode);
* **outcome-based**: they say *whether* an attempt worked, not *which* tokens were good or bad;
* **literal**: the model optimizes exactly what we check, not what we meant. If the format is part of the check, the format is part of what gets learned, and we'll see that happen in §2.

Spreading this single number back over every token of the response is what the rest of the pipeline is for.

**Beyond exact-match rewards.** Arithmetic can be checked in a few lines of Python, but many tasks can't ("is this explanation correct?"), and the reward then comes from a judge model, a test runner, or an external API. Nothing in the pipeline requires the reward to be cheap or local, but judges are noisier and easier to game than exact checks: the policy will learn a judge's blind spots just as literally as it learns the `\boxed{}` format here. We'll put an LLM judge next to an exact check in [notebook 4](4_rewards_and_judges.ipynb).

**Bundling into a module.** Later stages (Ray workers, verl's reward workers, the other notebooks in this series) run in other processes and need to import this code. It lives in `tutorial/lib/arith_task.py`, copied verbatim from the three cells above: nothing new.

In [8]:
from arith_task import BOXED, make_problems, reward_fn, safe_eval

# From here on we use the module versions. Quick sanity check:
TRAIN = make_problems(4000, seed=1)
EVAL = make_problems(200, seed=2)


print(f"{len(TRAIN)} training problems, {len(EVAL)} eval problems")
print("first problem:", TRAIN[0])

4000 training problems, 200 eval problems
first problem: {'question': 'What is 19 * 99 + 10?', 'answer': 1891}


In [9]:
TRAIN[:5], EVAL[:5]

([{'question': 'What is 19 * 99 + 10?', 'answer': 1891},
  {'question': 'What is 17 - 99 - 17?', 'answer': -99},
  {'question': 'What is 85 - 28 + 17?', 'answer': 74},
  {'question': 'What is 5 - 57 * 26?', 'answer': -1477},
  {'question': 'What is 2 * 59 - 25?', 'answer': 93}],
 [{'question': 'What is 9 + 12 - 28?', 'answer': -7},
  {'question': 'What is 23 * 87 - 10?', 'answer': 1991},
  {'question': 'What is 79 + 79 + 20?', 'answer': 178},
  {'question': 'What is 89 + 57 * 14?', 'answer': 887},
  {'question': 'What is 94 * 49 * 16?', 'answer': 73696}])

### 1.2 The policy, and sampling from it

In RL, the **policy** is whatever chooses the actions. For an LLM, an action is choosing the next token. Given everything written so far, the model outputs a probability for every token in its vocabulary (151,936 of them for Qwen2.5), and we sample the next token from that distribution. So the LLM *is* the policy. We write it as **π_θ**, where θ are the weights.

The model never sees text, only **token IDs**: integers that index its vocabulary. A tokenizer converts between the two. The cell below also defines `chat_prompt_ids`, which wraps a question in Qwen's **chat template**. The model was fine-tuned on conversations in one specific format, with special tokens marking who is speaking (`<|im_start|>user`, `<|im_start|>assistant`, ...), and it behaves like an assistant only when it sees that format. `add_generation_prompt=True` appends the opening of an assistant turn, so the next token the model writes is the start of its answer.

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

tokenizer = AutoTokenizer.from_pretrained(MODEL)
PAD_ID = tokenizer.pad_token_id
STOP_IDS = {tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.convert_tokens_to_ids("<|endoftext|>")}


def chat_prompt_ids(question: str) -> list[int]:
    """The question plus our instruction, wrapped in the model's chat template, as token IDs."""
    messages = [{"role": "user", "content": f"{question} {INSTRUCTION}"}]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True)


def decode_tokens(ids) -> str:
    return tokenizer.decode(ids, skip_special_tokens=True)

/home/azureuser/localfiles/agl-rl-tutorials/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


What does one of our problems look like after the chat template has been applied?

In [11]:
prompt_ids = chat_prompt_ids(TRAIN[0]["question"])
print(TRAIN[0]["question"])
print(f"{len(prompt_ids)} tokens: {prompt_ids[:10]} ...")
print()
print(decode_tokens(prompt_ids))

What is 19 * 99 + 10?
58 tokens: [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553] ...

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is 19 * 99 + 10? Show brief working, then give the final answer as \boxed{N}.
assistant



**Why sampling parameters matter for RL.** RL training assumes that every sampled token was drawn from π_θ, the model's actual distribution. Later, the loss computes log π_θ for those tokens and pushes them up or down. If the samples came from a *different* distribution, the update would be tuned for a model we don't have.

Chat models ship with sampling settings that change the distribution, because they read better in conversation. Let's see what this one ships with:

In [12]:
generation_config = GenerationConfig.from_pretrained(MODEL).to_dict()
sampling_keys = ("temperature", "top_p", "top_k", "repetition_penalty")
print("model's own sampling defaults:", {key: value for key, value in generation_config.items() if key in sampling_keys})
print("we override all four → temperature=1, top_p=1, top_k=0, repetition_penalty=1")

model's own sampling defaults: {'temperature': 0.7, 'top_k': 20, 'top_p': 0.8, 'repetition_penalty': 1.1}
we override all four → temperature=1, top_p=1, top_k=0, repetition_penalty=1


Each of these settings changes the distribution we sample from. Here is what two of them do to a toy 5-token vocabulary:

* `top_k` keeps only the k most likely tokens, and renormalises them so they sum to 1;
* `temperature` raises each probability to the power 1/T and renormalises, which sharpens the distribution when T < 1.

In [13]:
probs = torch.tensor([0.50, 0.20, 0.15, 0.10, 0.05])  # π_θ over a toy 5-token vocabulary

# top_k=2: keep only the 2 most likely tokens, then renormalise so they sum to 1
top2 = probs.topk(2)
top_k_probs = torch.zeros_like(probs)
top_k_probs[top2.indices] = top2.values
top_k_probs = top_k_probs / top_k_probs.sum()

# temperature=0.7: raise every probability to the power 1/0.7, then renormalise
temperature_probs = probs ** (1 / 0.7)
temperature_probs = temperature_probs / temperature_probs.sum()

print("token:                        ", [0, 1, 2, 3, 4])
print("π_θ (what the trainer scores):", [round(p, 3) for p in probs.tolist()])
print("sampled with top_k=2:         ", [round(p, 3) for p in top_k_probs.tolist()])
print("sampled with temperature=0.7: ", [round(p, 3) for p in temperature_probs.tolist()])

token:                         [0, 1, 2, 3, 4]
π_θ (what the trainer scores): [0.5, 0.2, 0.15, 0.1, 0.05]
sampled with top_k=2:          [0.714, 0.286, 0.0, 0.0, 0.0]
sampled with temperature=0.7:  [0.63, 0.17, 0.113, 0.063, 0.023]


With `top_k=2`, tokens 2, 3 and 4 can **never** be sampled, even though π_θ gives them 30% between them. The model would never learn whether they were good or bad choices. Token 0 shows up 71% of the time instead of 50%, so it looks more typical than it really is. Temperature 0.7 has a similar, milder effect. Either way, the trainer would be learning about a distribution the model doesn't actually have.

So for RL we switch all of them off: `temperature=1, top_p=1, top_k=0 (off), repetition_penalty=1`. With these settings, the sampled tokens really are draws from π_θ.

**Loading the policy.** Three settings in `load_policy` are about precision and memory.

**fp32 master weights.** We load the weights in full 32-bit precision (`dtype=torch.float32`), even though computing in 16-bit **bf16** is about twice as fast. The reason is the size of an update. With our learning rate of 2e-6, Adam changes each weight by roughly 0.000002 per step. bf16 keeps only about 3 significant digits, so near 1.0 the closest representable numbers are about 0.008 apart. Adding 0.000002 to a bf16 weight does nothing at all:

In [14]:
weight = torch.tensor(1.0)
update = 2e-6  # roughly the size of one Adam step with our learning rate

print("fp32 weight after the update:", (weight + update).item())
print("bf16 weight after the update:", (weight.to(torch.bfloat16) + update).item())
print("gap between neighbouring bf16 numbers near 1.0:", torch.finfo(torch.bfloat16).eps)

fp32 weight after the update: 1.0000020265579224
bf16 weight after the update: 1.0
gap between neighbouring bf16 numbers near 1.0: 0.0078125


In bf16 the update simply vanishes: every training step would round back to the same weights. So the weights the optimizer updates stay in fp32, and only the *computation* runs in bf16. Every forward pass is wrapped in `torch.autocast("cuda", dtype=torch.bfloat16)`, which runs the big matrix multiplications in bf16 for speed while the weights themselves stay fp32. You'll see that wrapper in `hf_generate` and `micro_batch_log_probs`. This is the same setup verl uses.

**Gradient checkpointing.** To compute gradients, the backward pass needs the intermediate results ("activations") of every layer from the forward pass. Storing them all for long sequences takes a lot of memory. With gradient checkpointing, the model stores only some of them and recomputes the rest during the backward pass: a bit slower, much less memory. `use_cache = False` turns off the key/value cache, which only speeds up token-by-token generation and is useless during training.

**FlashAttention-2** (`attn_implementation="flash_attention_2"`) computes attention in small blocks without ever building the full attention matrix, which is faster and uses far less memory for long sequences.

In [15]:
def load_policy(path: str = MODEL):
    hf_logging.set_verbosity_error()  # also inside Ray workers
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float32, attn_implementation="flash_attention_2")  # fp32 master weights
    model = model.to(DEVICE)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})  # recompute activations in backward: less memory
    model.config.use_cache = False  # the KV cache only speeds up generation, not training
    return model


def make_optimizer(model, lr: float):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)


policy = load_policy()
print(f"policy: {sum(param.numel() for param in policy.parameters()) / 1e6:.0f}M parameters")

policy: 494M parameters


**Generating samples.** `hf_generate` takes a list of prompts (as token-ID lists) and samples `samples_per_prompt` responses for each, step by step:

1. **Repeat each prompt** `samples_per_prompt` times, one row per sample. With 2 prompts and 3 samples each, the rows are `[p0, p0, p0, p1, p1, p1]`. This order is called **prompt-major**: all of prompt 0's samples come first, so sample *i* always belongs to prompt *i // samples_per_prompt*. We'll rely on that later to group attempts at the same question.
2. **Left-pad** the rows to the same length, so they fit in one tensor and every prompt ends at the same column, where generation starts. §1.3 explains the padding in detail.
3. **Generate**, sampling from the full distribution (the RL settings from above).
4. **Cut off the prompt columns**: `generate` returns prompt + response, and we only want the response.
5. **Truncate at the stop token.** Rows that finish early get filled with padding while the other rows keep generating. `truncate_at_stop` cuts each response just after its first stop token. We **keep** the stop token itself: deciding *when to stop* is part of what the model must learn, and the stop token is a token like any other, so it gets trained too.

In [ ]:
def hf_generate(model, prompts: list[list[int]], samples_per_prompt: int, max_new_tokens: int = 256) -> list[dict]:
    """Sample responses with plain `model.generate`. Samples are prompt-major: all of prompt 0's samples come first."""
    with torch.no_grad():
        
        # One row per sample: each prompt repeated samples_per_prompt times
        row_prompts = [prompt for prompt in prompts for _ in range(samples_per_prompt)]
        prompt_width = max(len(prompt) for prompt in row_prompts)  # every row is padded to the longest prompt

        # Left-pad, so that every prompt ends at column prompt_width, where generation starts
        input_ids = torch.full((len(row_prompts), prompt_width), PAD_ID, dtype=torch.long)
        attention_mask = torch.zeros_like(input_ids)
        
        for row, prompt in enumerate(row_prompts):
            num_padding = prompt_width - len(prompt)
            input_ids[row, num_padding:] = torch.tensor(prompt)
            attention_mask[row, num_padding:] = 1

        ### extract function here

        model.eval()
        

        with torch.autocast("cuda", dtype=torch.bfloat16):
            output_ids = model.generate(
                input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE),
                do_sample=True, temperature=1.0, top_p=1.0, top_k=0, repetition_penalty=1.0,
                max_new_tokens=max_new_tokens, pad_token_id=PAD_ID, use_cache=True,
            )

        model.train()

    # generate returns prompt + response; the response tokens start at column prompt_width
    response_ids = output_ids[:, prompt_width:].tolist()
    return [
        {"prompt_ids": prompt, "response_ids": truncate_at_stop(response), "log_probs": None}
        for prompt, response in zip(row_prompts, response_ids)
    ]


def truncate_at_stop(ids: list[int]) -> list[int]:
    """Cut a response just after its first stop token (generate pads rows that finished early)."""
    for position, token_id in enumerate(ids):
        if token_id in STOP_IDS:
            return ids[: position + 1]  # keep the stop token: the model must learn when to stop
    return ids

Let's generate 3 samples for the first training problem and see what we get. Each sample is a dict with `prompt_ids`, `response_ids`, and (for now) `log_probs=None`.

In [18]:
samples = hf_generate(policy, [chat_prompt_ids(TRAIN[0]["question"])], samples_per_prompt=3)

for i, sample in enumerate(samples):
    response_text = decode_tokens(sample["response_ids"])
    reward = reward_fn(response_text, TRAIN[0]["answer"])
    print(f"── sample {i}  (reward {reward}, {len(sample['response_ids'])} tokens) ──")
    print(response_text)
    print()

── sample 0  (reward 0.0, 102 tokens) ──
I apologize, but there's a mistake in the question. When you multiply 19 by 99 and then add 10, the calculation doesn't make sense in the context of a standard mathematical expression. The correct mathematical operations do work out to:

\[ 19 \times 99 + 10 = 1871 + 10 = 1881 \]

So, the final answer would be \(\boxed{1881}\).

── sample 1  (reward 1.0, 251 tokens) ──
To solve the expression \(19 \times 99 + 10\), let's go through the steps in detail:

1. **Multiply 19 by 99**:
   \[
   19 \times 99
   \]
   We can use the distributive property to simplify this:
   \[
   19 \times 99 = 19 \times (100 - 1) = 19 \times 100 - 19 \times 1 = 1900 - 19 = 1881
   \]

2. **Add 10 to the result**:
   \[
   1881 + 10
   \]
   To make the addition easier, we can break it down into simpler steps:
   \[
   1881 + 10
   \]
   First, add 10 to 1881:
   \[
   1881 + 10 = 1891
   \]

Therefore, the final answer is:
\[
\boxed{1891}
\]

── sample 2  (reward 0.0, 

Each sample dict carries the raw token IDs — that's what the trainer needs. Let's peek at the structure:

In [19]:
sample = samples[0]
print(f"prompt_ids:   {len(sample['prompt_ids'])} tokens  {sample['prompt_ids'][:8]}...")
print(f"response_ids: {len(sample['response_ids'])} tokens  {sample['response_ids'][:8]}...")
print(f"log_probs:    {sample['log_probs']}  (filled in later by a forward pass)")

prompt_ids:   58 tokens  [151644, 8948, 198, 2610, 525, 1207, 16948, 11]...
response_ids: 102 tokens  [40, 36879, 11, 714, 1052, 594, 264, 16523]...
log_probs:    None  (filled in later by a forward pass)


### 1.3 From samples to tensors

Our samples are Python lists of different lengths, but a GPU works on rectangular tensors. So we **pad** them: we fill the short ones with a padding token until every row has the same length, and use masks to tell the model which positions are real.

**The layout: prompts are left-padded, responses right-padded.** Take two samples: one with a 3-token prompt and a 4-token response, one with a 5-token prompt and a 2-token response (`P` = prompt token, `R` = response token, `·` = padding). `make_padded_batch` builds a prompt block as wide as the longest prompt and a response block as wide as the longest response:

```
              prompts (left-padded)   responses (right-padded)
              ┌───────────────────┐   ┌───────────────┐
  sample 0:   ·   ·   P   P   P   │   R   R   R   R
  sample 1:   P   P   P   P   P   │   R   R   ·   ·
                                  ↑
             column 5: every response starts here
```

`input_ids` is just the two blocks side by side.

**Why the prompts are padded on the left.** It lines up their *ends*, so the boundary between prompt and response is at the same column (`prompt_width`) in every row. Two things depend on that:

1. **Generation:** the model writes each new token straight after the last column. With left padding, every row continues directly from its real prompt rather than from padding.
2. **Reading out log-probs** (§1.4): every response sits in the same columns, so a single slice (the last `response_width` columns) gets every response in the batch. If the boundary were at a different column in each row, we'd need a different slice per row.

**Why the responses are padded on the right.** They already start at the same column, and right padding keeps it that way. It also puts the padding after each response's stop token, where it naturally belongs.

**Making the padding harmless.** Padding tokens are fake, so three things keep them from affecting anything:

* `attention_mask` (prompt + response): 1 for real tokens, 0 for padding, so the model never attends to padding.
* `response_mask` (response only): 1 for response tokens that should be **trained**. For now that's every real response token. In §7 the tool outputs inside a response will get 0.
* `position_ids`: each token's position among the *real* tokens of its row. Without them, sample 0's first real token would be treated as being at position 2 just because of the padding before it. The cell below shows how they're computed from the attention mask.

The key names (`prompts`, `responses`, `input_ids`, `attention_mask`, `position_ids`, `response_mask`) are verl's, which will make §3 a one-line change.

In [20]:
attention_mask = torch.tensor([
    [0, 0, 1, 1, 1, 1, 1, 1, 1],  # sample 0: 2 padding positions on the left
    [1, 1, 1, 1, 1, 1, 1, 0, 0],  # sample 1: 2 padding positions on the right
])
real_tokens_so_far = attention_mask.cumsum(-1)        # running count of real tokens, column by column
position_ids = (real_tokens_so_far - 1).clamp(min=0)  # first real token -> 0, the next -> 1, ...

print("attention_mask:\n", attention_mask)
print("cumsum:\n", real_tokens_so_far)
print("position_ids:\n", position_ids)

attention_mask:
 tensor([[0, 0, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0]])
cumsum:
 tensor([[0, 0, 1, 2, 3, 4, 5, 6, 7],
        [1, 2, 3, 4, 5, 6, 7, 7, 7]])
position_ids:
 tensor([[0, 0, 0, 1, 2, 3, 4, 5, 6],
        [0, 1, 2, 3, 4, 5, 6, 6, 6]])


In sample 0, the first real token gets position 0 even though it sits in column 2, so left padding doesn't shift the positions. The padding positions get leftover values (0 on the left, repeats on the right), but the attention mask hides them, so those values never matter.

In [21]:
def make_padded_batch(prompts: list[list[int]], responses: list[list[int]], response_masks: list[list[int]] | None = None) -> dict[str, torch.Tensor]:
    batch_size = len(prompts)
    prompt_width = max(len(prompt) for prompt in prompts)                       # the longest prompt
    response_width = max(1, max(len(response) for response in responses))     # the longest response (at least 1 column)

    prompt_ids = torch.full((batch_size, prompt_width), PAD_ID, dtype=torch.long)
    response_ids = torch.full((batch_size, response_width), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, prompt_width + response_width), dtype=torch.long)
    response_mask = torch.zeros((batch_size, response_width), dtype=torch.long)

    for row, (prompt, response) in enumerate(zip(prompts, responses)):
        prompt_start = prompt_width - len(prompt)  # prompts are left-padded...
        response_end = len(response)               # ...and responses right-padded
        prompt_ids[row, prompt_start:] = torch.tensor(prompt)
        response_ids[row, :response_end] = torch.tensor(response, dtype=torch.long)
        attention_mask[row, prompt_start : prompt_width + response_end] = 1
        trained = response_masks[row] if response_masks else [1] * len(response)  # default: train on every response token
        response_mask[row, :response_end] = torch.tensor(trained, dtype=torch.long)

    return {
        "prompts": prompt_ids,
        "responses": response_ids,
        "input_ids": torch.cat([prompt_ids, response_ids], dim=1),
        "attention_mask": attention_mask,
        # each token's position among the row's real tokens (0, 1, 2, ...), so left-padding doesn't shift positions
        "position_ids": (attention_mask.cumsum(-1) - 1).clamp(min=0),
        "response_mask": response_mask,
    }

Let's batch up our three samples with `make_padded_batch` and check the shapes:

In [23]:
batch = make_padded_batch([s["prompt_ids"] for s in samples], [s["response_ids"] for s in samples])
for k, v in batch.items():
    print(f"{k:15} {tuple(v.shape)}")

prompts         (3, 58)
responses       (3, 251)
input_ids       (3, 309)
attention_mask  (3, 309)
position_ids    (3, 309)
response_mask   (3, 251)


The shapes make sense: 3 rows; `prompts` is as wide as the longest prompt (`prompt_width`), `responses` as wide as the longest response (`response_width`), and `input_ids` is the two side by side. But what does the padding actually look like? Let's visualise one row and see where the real tokens sit vs. the padding:

In [24]:
row = 0
prompt_len = len(samples[row]["prompt_ids"])
response_len = len(samples[row]["response_ids"])
prompt_width = batch["prompts"].shape[1]
response_width = batch["responses"].shape[1]

print(f"row {row}: prompt has {prompt_len} tokens (padded to {prompt_width}), response has {response_len} tokens (padded to {response_width})")
print(f"         input_ids width = {prompt_width} + {response_width} = {prompt_width + response_width}")
print()


def as_blocks(mask_values) -> str:
    return "".join("█" if value else "·" for value in mask_values)


# Show the mask pattern
attention_mask = batch["attention_mask"][row].tolist()
response_mask = batch["response_mask"][row].tolist()
shown = min(response_len + 5, response_width)  # the response plus a few padding positions
for label, values in [
    ("attention_mask (prompt region)", attention_mask[:prompt_width]),
    ("attention_mask (response region)", attention_mask[prompt_width : prompt_width + shown]),
    ("response_mask", response_mask[:shown]),
]:
    print(f"{label + ':':35}", as_blocks(values))  # pad every label to the same width, so the rows line up
print()
print("█ = real token, · = padding")
print(f"Left-pad in prompt:  {prompt_width - prompt_len} positions")
print(f"Right-pad in response: {response_width - response_len} positions")

row 0: prompt has 58 tokens (padded to 58), response has 102 tokens (padded to 251)
         input_ids width = 58 + 251 = 309

attention_mask (prompt region):     ██████████████████████████████████████████████████████████
attention_mask (response region):   ██████████████████████████████████████████████████████████████████████████████████████████████████████·····
response_mask:                      ██████████████████████████████████████████████████████████████████████████████████████████████████████·····

█ = real token, · = padding
Left-pad in prompt:  0 positions
Right-pad in response: 149 positions


### 1.4 Log-probs: the currency of policy gradients

Some notation, since we’ll use it everywhere from here on. **x** is the prompt, and **y = (y₁, …, y_T)** is the response, T tokens long. **π_θ(y_t | x, y_{<t})** is the probability the model gives to token y_t, given the prompt and the response so far. The model generates one token at a time, so the probability of a whole response is a product, and its log is a sum:

$$\log \pi_\theta(y \mid x) = \sum_{t=1}^{T} \log \pi_\theta(y_t \mid x, y_{<t})$$

Almost everything an RL trainer computes is built from these per-token **log-probs**, for a few reasons:

* **Numerics.** A 300-token response might have a probability like 0.9³⁰⁰ ≈ 10⁻¹⁴. Multiplying hundreds of numbers below 1 underflows; adding their logs doesn’t.
* **The gradient comes out in logs.** The policy-gradient theorem says the direction that increases the expected reward is ∇ log π_θ(y | x), weighted by how good y was. That’s what we’ll compute in §1.5.
* **Per-token credit.** Because the log of a response is a *sum*, every token gets its own term, so we can train some tokens and skip others just by masking them out of the sum. We’ll need this for tool outputs in §7.
* **Ratios become differences.** PPO compares the new policy with the old one through π_θ / π_old = exp(log π_θ − log π_old).

It’s also the quantity SFT uses: the cross-entropy loss is just −log π_θ of the demonstration tokens. SFT pushes up the log-probs of someone else’s text; RL pushes the log-probs of the model’s *own* samples up or down, depending on how they scored.

First, a tiny indexing helper. `select_rows` slices a batch by row, whether it’s a plain dict of tensors or the `DataProto` we’ll meet in §3.

In [ ]:
def select_rows(batch, index):
    """Index a batch given as a dict of tensors, or as verl's TensorDict (§3)."""
    if isinstance(batch, dict):
        return {key: tensor[index] for key, tensor in batch.items()}
    return batch[index]

**`micro_batch_log_probs`: how likely was each response token?** For every response token, we want the answer to one question: *how likely did the model think the token we actually have was?* That single number per token, log π_θ(y_t | x, y_<t), is what the loss pushes up or down.

The model answers a bigger question than that. At every position it gives a probability for **every** possible next token, all 151,936 of them. But we already know which token is there (it was sampled), so we only need the one entry for that token. Getting it takes four steps:

1. Run **one forward pass** over prompt + response. A causal LLM produces its predictions for every position in parallel, so scoring a response that already exists needs no token-by-token generation.
2. **Slice** to the positions that predict response tokens. The prediction made *at* position t−1 is the prediction *for* token t, so we take the last prompt position plus every response position except the final one.
3. **`log_softmax`** turns raw scores ("logits") into log-probabilities over the whole vocabulary.
4. **`gather`** picks out, at each position, the log-prob of the token that's actually there.

Here are steps 2–4 on a toy example: a 5-token vocabulary (ids 0–4), one row, and the 3-token response `[3, 1, 4]`. We asked the model for `logits_to_keep = response_width + 1 = 4` positions: the last prompt token plus the 3 response tokens.

In [ ]:
response_ids = torch.tensor([[3, 1, 4]])

# What the model returns with logits_to_keep = 4: at each kept position, one score per vocabulary token
logits = torch.tensor([[[0.1, 0.2, 0.3, 2.0, 0.0],    # at the last prompt token -> predicts response token 0
                        [0.0, 1.5, 0.2, 0.1, 0.3],    # at response token 0      -> predicts response token 1
                        [0.2, 0.1, 0.0, 0.3, 1.0],    # at response token 1      -> predicts response token 2
                        [0.9, 0.9, 0.9, 0.9, 0.9]]])  # at response token 2      -> predicts the token AFTER the response
print("logits:", tuple(logits.shape), "= (rows, positions, vocabulary)\n")

# Step 2: drop the last position: it predicts a token that isn't part of the response
logits = logits[:, :-1].float()
print("step 2, logits[:, :-1]:", tuple(logits.shape), "\n")

# Step 3: scores -> log-probabilities over the vocabulary, separately at each position
log_probs = torch.log_softmax(logits, dim=-1)
print("step 3, log_softmax:\n", log_probs.round(decimals=2))
print("each position's probabilities sum to:", log_probs.exp().sum(-1).round(decimals=2).tolist(), "\n")

# Step 4: at each position, keep only the log-prob of the token that is actually there
token_log_prob = log_probs.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1)
print("step 4, gather:", [round(x, 2) for x in token_log_prob[0].tolist()])

Reading the output step by step:

* **Step 2: `logits[:, :-1]`** means "all rows, every position except the last". The last position predicts what comes *after* the response, and there's no such token to score. What's left lines up exactly with the response: row *t* holds the prediction for response token *t*. (`.float()` converts from bf16 to fp32, so the next step is numerically precise.)
* **Step 3: `log_softmax`** turns each row of raw scores into log-probabilities. `dim=-1` means "across the vocabulary", separately at each position. Exponentiated, each row sums to 1: at position 0, for example, the model gives token 3 a probability of e^−0.49 ≈ 0.61.
* **Step 4: `gather`** is a lookup. At each position it goes to the column of the token that's actually there and reads off that one value:

```
                      tok0   tok1   tok2   tok3   tok4    actual token   value taken
predicts token 0   [ -2.39, -2.29, -2.19, -0.49, -2.49 ]        3       →  -0.49
predicts token 1   [ -2.21, -0.71, -2.01, -2.11, -1.91 ]        1       →  -0.71
predicts token 2   [ -1.80, -1.90, -2.00, -1.70, -1.00 ]        4       →  -1.00
```

`gather(-1, index)` reads as "along the last dimension (the vocabulary), pick the entry at `index`". It insists that the index has as many dimensions as the table, which is why `unsqueeze(-1)` turns `[[3, 1, 4]]` into `[[[3], [1], [4]]]` first, and `squeeze(-1)` removes that extra dimension afterwards. The lookup itself is nothing more than this double loop, done in one GPU operation:

In [ ]:
looked_up = torch.zeros(1, 3)
for row in range(1):
    for position in range(3):
        token_id = response_ids[row, position]                         # which token is actually here
        looked_up[row, position] = log_probs[row, position, token_id]  # read that column of the row

print("double loop:", [round(x, 2) for x in looked_up[0].tolist()])
print("gather:     ", [round(x, 2) for x in token_log_prob[0].tolist()])

**Why that one number is all RL needs.** Everything the trainer does is built from it:

* **Adding them up** gives log π(response), how likely the model was to write this exact response: here −0.49 − 0.71 − 1.00 = −2.20, a probability of e^−2.2 ≈ 0.11.
* **The update** pushes these numbers up for responses with a positive advantage and down for negative ones.
* **PPO's ratio** (§1.5) compares this same number before and after an update, token by token.

None of this needs the probabilities of tokens the model *didn't* write. (Raising the chosen token's probability lowers the others automatically, because softmax keeps each row summing to 1.)

**Why `compute_log_probs` works in micro-batches.** Before the slicing and lookup, the logits are huge: one score per vocabulary token, at every position, for every row. For one row with a 256-token response, that's 257 × 151,936 numbers: about 78 MB in bf16, 156 MB after `.float()`, and another 156 MB for the `log_softmax` output, so roughly 0.4 GB per row. From §2 on, a training step has 128 rows, which would mean about 50 GB just to read off one number per token. Splitting the batch into micro-batches of 16 rows needs about 6 GB at a time, freed before the next micro-batch. The result is identical, because each row's log-probs depend only on that row: the attention mask keeps rows separate.

In [ ]:
def micro_batch_log_probs(model, micro_batch, with_grad: bool = False) -> torch.Tensor:
    """log π(token) for every response token; shape (rows, response_width)."""
    response_ids = micro_batch["responses"].to(DEVICE)
    response_width = response_ids.shape[1]
   
    with torch.set_grad_enabled(with_grad), torch.autocast("cuda", dtype=torch.bfloat16):
        logits = model(  # 1. one forward pass over prompt + response
            input_ids=micro_batch["input_ids"].to(DEVICE),
            attention_mask=micro_batch["attention_mask"].to(DEVICE),
            position_ids=micro_batch["position_ids"].to(DEVICE),
            logits_to_keep=response_width + 1,
        ).logits
    
    logits = logits[:, :-1].float()                # 2. position t-1 predicts token t, so drop the last position
    log_probs = torch.log_softmax(logits, dim=-1)  # 3. log-probs over the whole vocabulary
    token_log_prob = log_probs.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1) # 4. the log-prob of the token actually there
    return token_log_prob  


def compute_log_probs(model, batch, micro_batch_size: int = 16) -> torch.Tensor:
    with torch.no_grad():
        model.eval()
        num_rows = batch["input_ids"].shape[0]
        chunks = []
        for start in range(0, num_rows, micro_batch_size):
            micro_batch = select_rows(batch, slice(start, start + micro_batch_size))
            chunks.append(micro_batch_log_probs(model, micro_batch))
    
    model.train()  # back to training mode, so ppo_update gets gradient checkpointing again
    return torch.cat(chunks).cpu()

Let’s compute log-probs for our batch and see what the numbers look like for one sample:

In [ ]:
log_probs = compute_log_probs(policy, batch)
is_response_token = batch["response_mask"][0].bool()
sample0_log_probs = log_probs[0][is_response_token]  # sample 0's real response tokens, no padding

print(f"sample 0: {is_response_token.sum()} response tokens")
print(f"  log-probs: min {sample0_log_probs.min():.2f}, max {sample0_log_probs.max():.2f}, mean {sample0_log_probs.mean():.2f}")
print(f"  sum log π(response) = {sample0_log_probs.sum():.1f}  (i.e. π ≈ e^{sample0_log_probs.sum():.0f})")
print(f"  first 10 token log-probs: {[round(x, 2) for x in sample0_log_probs[:10].tolist()]}")

How to read these numbers:

* **max ≈ 0:** some tokens are almost certain (log p ≈ 0 means p ≈ 1), like the obvious continuation of a number or a formatting token.
* **min, a large negative number:** at least one token was very surprising. It was sampled anyway, because we sample from the full distribution at temperature 1.
* **mean, close to 0:** a typical token is quite likely; the model is confident most of the time.
* **the sum:** the log-probability of the whole response, because the probability of a sequence is the product of its token probabilities. It looks absurdly small (e^−57 or so), but that's normal: there are astronomically many possible responses of this length. The absolute value means little. What RL works with is how it **changes**, and §1.7 checks exactly that.

To get a feel for these numbers, let’s ask the policy about a question with an easy-to-get-wrong answer. First, the distribution over the very first response token: the logits at the *last prompt position*, through a softmax. This is π_θ(y₁ | x), the distribution the first token is sampled from.

In [ ]:
demo_prompt = chat_prompt_ids("What is 2 + 3 * 4?")

with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    logits = policy(torch.tensor([demo_prompt], device=DEVICE)).logits

next_token_probs = torch.softmax(logits[0, -1].float(), dim=-1)  # the last prompt position predicts the first response token
top5 = next_token_probs.topk(5)

print("π(first response token | prompt), top 5:")

for prob, token_id in zip(top5.values.tolist(), top5.indices.tolist()):
    print(f"   {tokenizer.decode([token_id])!r:>10}  p = {prob:.3f}   log p = {np.log(prob):+.3f}")

Now let’s score three **hand-written** responses. The model never generated them: we *teacher-force* each one, feeding it in and reading off the log-prob of each of its tokens. That answers “if we sampled from the model, how likely would we be to get exactly this text?”, so we can score any string, including ones the model would rarely write.

* the correct working, ending in `\boxed{14}`;
* the classic mistake, adding before multiplying, ending in `\boxed{20}`;
* the right number, but without the `\boxed{}` our reward requires.

Each one ends with the `<|im_end|>` stop token, because the model must also learn *when to stop*.

In [ ]:
candidates = [r"2 + 3 * 4 = 2 + 12 = 14. \boxed{14}", r"2 + 3 * 4 = 5 * 4 = 20. \boxed{20}", "The answer is 14."]

im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")

candidate_ids = [tokenizer.encode(text, add_special_tokens=False) + [im_end] for text in candidates]
candidate_batch = make_padded_batch([demo_prompt] * len(candidate_ids), candidate_ids)
candidate_log_probs = compute_log_probs(policy, candidate_batch) * candidate_batch["response_mask"]

for text, ids, per_token in zip(candidates, candidate_ids, candidate_log_probs):
    per_token = per_token[: len(ids)]  # drop the padding
    least_likely = int(per_token.argmin())
    print(f"{text!r:42} reward {reward_fn(text, 14):.0f} | log π(response) = {per_token.sum():7.2f} over {len(ids):2d} tokens"
          f" | least likely token {tokenizer.decode([ids[least_likely]])!r} (log p {per_token[least_likely]:.2f})")

The correct working is the most likely of the three. The order-of-operations slip is less likely by a factor of about e⁸ ≈ 3,000, and most of that comes from a single token: the `5` in `5 * 4`, exactly where the mistake happens. The bare answer is only 8 tokens long, but after an instruction that asked for working, each of its tokens is unlikely, so its total is still lower than the correct working’s. It’s also worth noticing which token of the correct response is least likely: the ` \` that starts `\boxed`. The model isn’t sure yet that it should use the format we reward, and we’ll see RL fix that in §2.

This is the same scoring SFT and RL both rely on. The whole job of RL training is to change these numbers: raise the log-probs of responses that turned out better than expected, and lower the ones that turned out worse. “Better than expected” is the next piece.

### 1.5 Advantages and the loss

The basic policy gradient is

$$\nabla_\theta J = \mathbb{E}\Big[\, A \cdot \nabla_\theta \log \pi_\theta(y \mid x) \Big],$$

that is, push up the log-prob of a response in proportion to its **advantage** A: how much better it did than expected.

**Why not just use the reward as the advantage?** Take two questions, an easy one the model gets right 4 times out of 4, and a hard one it gets right 1 time out of 4:

| | easy question: rewards | hard question: rewards |
|---|---|---|
| advantage = reward | +1, +1, +1, +1 | +1, 0, 0, 0 |
| advantage = reward − group mean | 0, 0, 0, 0 | +0.75, −0.25, −0.25, −0.25 |

With the raw reward, every correct answer gets the same push, whether it was a routine success or a rare breakthrough, and the wrong answers (reward 0) aren't pushed down at all. Subtracting what we **expected** (the **baseline**, here the group's mean reward) fixes both: the easy question's attempts teach nothing new, the rare success gets pushed up, and the failures get pushed down. It also makes the gradient much less noisy.

Classic PPO trains a separate *critic* network to predict that baseline. GRPO drops the critic and uses the group itself: with n attempts at the same question,

$$A_i = \frac{r_i - \operatorname{mean}(r_1, \dots, r_n)}{\operatorname{std}(r_1, \dots, r_n) + \varepsilon}$$

Dividing by the standard deviation rescales each group so its advantages have a standard deviation of 1; we'll look at what that does below. Every response token of attempt *i* gets the same A_i. Here's the formula worked through for one group:

In [ ]:
rewards = torch.tensor([1.0, 0.0, 0.0, 0.0])  # one group: 4 attempts at one question, 1 correct

mean = rewards.mean()  # the baseline: how well the model does on this question, on average
std = rewards.std()    # how spread out the rewards are (torch's default divides by n - 1)
advantages = (rewards - mean) / (std + 1e-6)

print("rewards:       ", rewards.tolist())
print("mean:          ", round(mean.item(), 3))
print("rewards - mean:", (rewards - mean).tolist())
print("std:           ", round(std.item(), 3))
print("advantages:    ", [round(a, 2) for a in advantages.tolist()])

`grpo_advantages` below does exactly this, for every group in a batch at once: `group_ids` says which group each reward belongs to. Let's run it on a few groups of four 0/1 rewards:

In [ ]:
def grpo_advantages(rewards: torch.Tensor, group_ids, eps: float = 1e-6) -> torch.Tensor:
    """Each reward minus its group's mean, divided by its group's std. A group is all the attempts at one question."""
    
    advantages = torch.zeros_like(rewards)
    group_ids = np.asarray(group_ids)
    
    for group in np.unique(group_ids):
        members = torch.as_tensor(np.nonzero(group_ids == group)[0])  # row indices of this group's attempts
        group_rewards = rewards[members]
        
        if len(members) > 1:
            advantages[members] = (group_rewards - group_rewards.mean()) / (group_rewards.std() + eps)
    
    return advantages


groups = {"1 of 4 right": [1, 0, 0, 0], "3 of 4 right": [1, 1, 1, 0], "all right": [1, 1, 1, 1], "all wrong": [0, 0, 0, 0]}

# Flatten into one batch of 16 rewards, labelling each with the group it came from
rewards = torch.tensor([reward for group_rewards in groups.values() for reward in group_rewards], dtype=torch.float32)
group_ids = [name for name, group_rewards in groups.items() for _ in group_rewards]

demo_adv = grpo_advantages(rewards, group_ids)

for (name, group_rewards), group_adv in zip(groups.items(), demo_adv.split(4)):
    print(f"{name:13} rewards {group_rewards} -> advantages {[round(a, 2) for a in group_adv.tolist()]}")

Two things to notice. A rare success (1 of 4) gets a big push, much bigger than each success in a group that mostly succeeds: that’s the “better than expected” part. And groups where every attempt got the same reward have **zero advantage**: they contribute nothing to the gradient, but still cost all the compute to generate. With a task that’s too easy or too hard for the model, most groups look like that, which is why we chose a task the model sometimes gets right.

**How group size changes the signal.** A group only produces a learning signal if its attempts *disagree*. If the model succeeds 30% of the time, a group of n attempts is all wrong with probability 0.7ⁿ:

| group size n | P(all wrong) = 0.7ⁿ | P(all right) = 0.3ⁿ |
|---|---|---|
| 4 | 24% | 0.8% |
| 8 | 5.8% | ~0% |
| 16 | 0.3% | ~0% |

So bigger groups waste fewer questions. Let's check by simulating 50 questions at each group size:

In [ ]:
num_questions, success_rate = 50, 0.3
for group_size in [4, 8, 16]:
    # Simulate 50 questions with group_size attempts each, where every attempt succeeds 30% of the time
    rng = random.Random(42)
    
    rewards = torch.tensor([float(rng.random() < success_rate) for _ in range(num_questions * group_size)])
    group_ids = [i // group_size for i in range(num_questions * group_size)]  # attempts 0..group_size-1 are question 0, and so on
    advantages = grpo_advantages(rewards, group_ids)
    zero_groups = 0
    
    for question in range(num_questions):
        group_advantages = advantages[question * group_size : (question + 1) * group_size]
        if group_advantages.abs().sum() == 0:
            zero_groups += 1
    print(f"n={group_size:2d}: {zero_groups}/{num_questions} groups have zero advantage (all same reward) → {zero_groups / num_questions:.0%} wasted")

The simulation roughly matches the table (with only 50 questions, the counts are noisy). But each question costs n times as many tokens to generate, so bigger groups are not free: n = 8 is a common default.

**With vs. without std normalization ("Dr. GRPO").** Dividing by the group's standard deviation doesn't change anything *within* a group: every advantage in the group is divided by the same number, so the ordering and the ratios stay the same. What it changes is how groups are weighted **against each other**. The std is smallest for lopsided groups (1 right out of 8, or 7 right out of 8) and largest for evenly split ones (4 out of 8). So dividing by it scales up the advantages of questions the model almost always gets wrong, or almost always gets right.

The "Dr. GRPO" paper ([Liu et al., 2025](https://arxiv.org/abs/2503.20783)) argues this is a bias toward questions that are too easy or too hard, and simply drops the division. Let's compare the two on three groups of 8:

In [ ]:
def grpo_advantages_no_std(rewards, group_ids):
    """Like grpo_advantages, but without dividing by std (the Dr. GRPO variant)."""
    advantages = torch.zeros_like(rewards)
    group_ids = np.asarray(group_ids)
    for group in np.unique(group_ids):
        members = torch.as_tensor(np.nonzero(group_ids == group)[0])
        advantages[members] = rewards[members] - rewards[members].mean()
    return advantages


groups = {
    "1 of 8 right": [1, 0, 0, 0, 0, 0, 0, 0],
    "4 of 8 right": [1, 1, 1, 1, 0, 0, 0, 0],
    "7 of 8 right": [1, 1, 1, 1, 1, 1, 1, 0],
}
rewards = torch.tensor([reward for group_rewards in groups.values() for reward in group_rewards], dtype=torch.float32)
group_ids = [name for name, group_rewards in groups.items() for _ in group_rewards]

adv_std = grpo_advantages(rewards, group_ids)
adv_no_std = grpo_advantages_no_std(rewards, group_ids)

print(f"{'':13}  {'std':>5}   {'with std (GRPO)':>20}   {'without std (Dr. GRPO)':>22}")
print(f"{'':13}  {'':>5}   {'right':>9} {'wrong':>10}   {'right':>11} {'wrong':>10}")
for (name, group_rewards), with_std, without_std in zip(groups.items(), adv_std.split(8), adv_no_std.split(8)):
    group_std = torch.tensor(group_rewards, dtype=torch.float32).std()
    right = group_rewards.index(1)  # position of one correct attempt in the group
    wrong = group_rewards.index(0)  # position of one wrong attempt
    print(f"{name:13}  {group_std:5.2f}   {with_std[right]:+9.2f} {with_std[wrong]:+10.2f}   {without_std[right]:+11.2f} {without_std[wrong]:+10.2f}")

Reading across the rows:

* **Within a group, both versions agree on the direction and the ratios.** In the 1-of-8 group, the success gets 7 times the push of each failure, with or without std (+2.47 vs −0.35, or +0.88 vs −0.12).
* **Across groups, std changes the weighting.** The lopsided groups (1 of 8 and 7 of 8) have std 0.35, so with std their advantages are multiplied by about 2.8. The even group has std 0.53, so its advantages are multiplied by only about 1.9. With std, the rare success in the 1-of-8 group gets +2.47; without it, +0.88.

Whether that's good depends on the task. Amplifying lopsided groups helps the model learn from rare successes on hard questions, but it also amplifies the noise from questions that are nearly solved. We use plain GRPO (with std), as verl does by default.

**The loss.** We could now minimise −Σ A·log π directly. But generating samples is expensive, so real trainers reuse each batch for several optimizer steps (several mini-batches, sometimes several epochs). After the first step, the weights θ no longer match the weights θ_old that generated the samples, and we need a way to account for that, and to stop one batch from moving the policy too far.

PPO does this per token with the probability **ratio** ρ = π_θ / π_old ("how much more likely is this token now than when it was sampled?") and **clips** it:

$$L_t = -\min\!\big(\rho_t A_t,\; \operatorname{clip}(\rho_t, 1-\varepsilon_\text{low}, 1+\varepsilon_\text{high})\, A_t\big)$$

In words: the objective ρ·A rewards raising good tokens (A > 0) and lowering bad ones (A < 0), but once a token has moved far enough in the direction its advantage wants, the clipped term takes over. It's flat, so the gradient for that token becomes zero, and the batch can't push it any further. This keeps each batch's update within a "trust region" around the old policy. We use ε_low = 0.2 and ε_high = 0.28, the asymmetric "clip-higher" trick from DAPO, which leaves more room to raise unlikely good tokens.

Here is the formula applied to a single token that had probability 0.10 when it was sampled, as the new policy makes it more or less likely:

In [ ]:
old_prob = 0.10  # π_old: the token's probability when it was sampled
clip_low, clip_high = 0.2, 0.28

for advantage in [+1.0, -1.0]:
    print(f"advantage A = {advantage:+.0f}")
    print(f"  {'new prob':>8} {'ratio ρ':>8} {'ρ·A':>7} {'clip(ρ)·A':>10} {'objective':>10}  gradient")
    
    for new_prob in [0.05, 0.09, 0.10, 0.12, 0.15, 0.30]:
        ratio = new_prob / old_prob
        unclipped = ratio * advantage
        clipped = min(max(ratio, 1 - clip_low), 1 + clip_high) * advantage
        objective = min(unclipped, clipped)  # the more pessimistic of the two
        gradient = "zero (clipped)" if clipped < unclipped else "flows"
        print(f"  {new_prob:8.2f} {ratio:8.2f} {unclipped:+7.2f} {clipped:+10.2f} {objective:+10.2f}  {gradient}")
    print()

Reading the A = +1 table: while the token's probability rises from 0.10 to 0.128 (ratio 1.28), the objective keeps growing, so the gradient keeps pushing it up. Beyond that (0.15, 0.30) the clipped term is smaller, `min` picks it, and since it's a constant the gradient is zero: this batch has pushed the token far enough. If the token moved the *wrong* way (0.05, 0.09), the unclipped term is the smaller one, so the gradient still flows and can undo the move.

The A = −1 table is the mirror image: lowering the token helps until the ratio reaches 0.8 (probability 0.08), then the gradient stops, as at 0.05. If a bad token became *more* likely (0.12, 0.15, 0.30), the gradient keeps flowing to push it back down.

`ppo_clip_loss` below computes this for every token at once. It returns `max` of the two *losses* (−ρA and −clip(ρ)A), which is the same thing as `min` of the two objectives with the sign flipped.

In [ ]:
def ppo_clip_loss(log_probs, old_log_probs, advantages, mask, num_trained_tokens, clip_low: float = 0.2, clip_high: float = 0.28):
    ratio = torch.exp(log_probs - old_log_probs)  # ρ = π_θ / π_old, per token
    
    unclipped = -advantages * ratio
    
    clipped = -advantages * ratio.clamp(1 - clip_low, 1 + clip_high)
    
    loss = torch.maximum(unclipped, clipped)  # the larger loss = the smaller (more pessimistic) objective
    
    return (loss * mask).sum() / num_trained_tokens     # mean over trained tokens; num_trained_tokens counts the whole mini-batch (see ppo_update)

Let’s plot the clipped objective for positive and negative advantages to see what the clip does:

In [ ]:
ratio = torch.linspace(0.3, 2.5, 500)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, advantage in zip(axes, [1.0, -1.0]):
    objective = -torch.maximum(-advantage * ratio, -advantage * ratio.clamp(0.8, 1.28))  # minus ppo_clip_loss, for one token
    ax.plot(ratio, advantage * ratio, "--", color="grey", label="ρ·A (unclipped)")
    ax.plot(ratio, objective, lw=2.5, label="PPO-clip objective")
    is_flat = np.isclose(np.gradient(objective.numpy(), ratio.numpy()), 0)
    ax.fill_between(ratio, objective.min() - 0.2, objective.max() + 0.2, where=is_flat, color="orange", alpha=0.15, label="zero gradient")
    ax.axvline(1, color="k", lw=0.5)
    ax.set_title(f"advantage A = {advantage:+.0f}")
    ax.set_xlabel("ratio ρ = π_θ / π_old")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

With a positive advantage (left), raising the token’s probability helps until ρ = 1.28, after which there’s nothing more to gain from this batch. With a negative advantage (right), lowering it helps until ρ = 0.8.

**`ppo_update`: running the optimizer.** This loops over the batch at two levels:

* A **mini-batch** is the set of rows that make up one optimizer step. In §1, a step has 8 questions × 8 samples = 64 rows and `mini_batch_size=64`, so the whole batch is one optimizer step. From §2 on, a step has 128 rows, so the same batch gives two optimizer steps: this is where the PPO ratio stops being exactly 1.
* A **micro-batch** is the set of rows that fit in GPU memory for one forward and backward pass (8 here). Calling `.backward()` once per micro-batch *adds* to the gradients, so after all of a mini-batch's micro-batches, the gradients are the same as for one big pass over the mini-batch. This is called **gradient accumulation**.

For that to work exactly, every micro-batch must divide its loss by the number of trained tokens in the **whole mini-batch**, not by its own token count. Here's why, with two micro-batches of different sizes:

In [ ]:
micro_batch_a = torch.tensor([1.0, 1.0, 1.0])  # per-token losses: 3 trained tokens
micro_batch_b = torch.tensor([4.0])            # 1 trained token
num_trained_tokens = 4                         # in the whole mini-batch

true_token_mean = torch.cat([micro_batch_a, micro_batch_b]).mean()
own_count = micro_batch_a.mean() + micro_batch_b.mean()  # each divides by its own count; backward adds them
whole_count = micro_batch_a.sum() / num_trained_tokens + micro_batch_b.sum() / num_trained_tokens  # what ppo_clip_loss does

print("mean over all 4 tokens (what we want):          ", true_token_mean.item())
print("each micro-batch divided by its own token count:", own_count.item())
print("each divided by the whole mini-batch's count:   ", whole_count.item())

Dividing by each micro-batch's own count gives 5.0: the single token in micro-batch B counts as much as all three tokens in A together. Dividing by the whole mini-batch's count gives exactly the token mean, 1.75. That's why `ppo_update` counts `num_trained_tokens` for the mini-batch first and passes it into every micro-batch's loss.

After each mini-batch, `clip_grad_norm_` scales the gradients down if their overall size exceeds 1.0, a safety net against a single unusual batch causing a huge step. Then the optimizer steps and the gradients are reset.

In [ ]:
def ppo_update(model, optimizer, batch, old_log_probs, advantages, mini_batch_size: int = 64, micro_batch_size: int = 8, loss_fn=ppo_clip_loss) -> dict:
    """One pass over the batch: one optimizer step per mini-batch, gradient accumulation over micro-batches."""
    num_rows = batch["input_ids"].shape[0]
    grad_norms = []
    for mini_start in range(0, num_rows, mini_batch_size):
        mini_end = min(mini_start + mini_batch_size, num_rows)
        num_trained_tokens = batch["response_mask"][mini_start:mini_end].sum().item()  # trained tokens in the whole mini-batch

        # Accumulate gradients over micro-batches small enough to fit in memory
        for micro_start in range(mini_start, mini_end, micro_batch_size):
            row_slice = slice(micro_start, min(micro_start + micro_batch_size, mini_end))
            micro_batch = select_rows(batch, row_slice)
            log_probs = micro_batch_log_probs(model, micro_batch, with_grad=True)
            mask = micro_batch["response_mask"].to(DEVICE).float()
            loss = loss_fn(log_probs, old_log_probs[row_slice].to(DEVICE), advantages[row_slice].to(DEVICE), mask, num_trained_tokens)
            loss.backward()

        # Then one optimizer step for the mini-batch
        grad_norms.append(torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0).item())
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    return {"grad_norm": float(np.mean(grad_norms))}

### 1.6 The training step

`rl_step` is the whole RL loop for one step: **sample → score → batch → old log-probs → advantages → update**. Following one §1 step (8 questions × 8 attempts) through it:

| Step | What happens | Result |
|---|---|---|
| 1. sample | `generate` writes 8 attempts at each of the 8 questions | 64 samples, prompt-major |
| | `group_ids = [i // 8 ...]` | `[0,0,0,0,0,0,0,0, 1,1,...,7]`: which question each sample answers |
| 2. score | `reward_fn` on each decoded response | `rewards`, shape (64,), each 0 or 1 |
| 3. batch | `make_padded_batch` | tensors with 64 rows: `responses` is (64, response_width) |
| 4. old log-probs | `compute_log_probs` with the current weights | `old_log_probs`, shape (64, response_width): π_old for the ratio |
| 5. advantages | `grpo_advantages(rewards, group_ids)` | one number per sample, shape (64,) |
| | `[:, None] * response_mask` | copied onto every trained token: shape (64, response_width), 0 on padding |
| 6. update | `ppo_update` | one optimizer step (64 rows = one mini-batch) |

`rl_step` takes the generation function as an argument. That's what lets later sections swap in a different inference engine without touching anything else.

If the inference engine returns its own log-probs for the sampled tokens (HF `generate` doesn't, vLLM will), `rl_step` also reports `logp_mismatch`: the mean |log π_trainer − log π_engine| over trained tokens. Both should describe the same model, so it should be small, and in §2 it tells us whether weight syncing works.

In [ ]:
def rl_step(policy, optimizer, generate, problems, samples_per_prompt: int = 8, **update_kwargs) -> tuple[dict, dict, list[dict]]:
    timing = {}

    # 1. Sample several responses per problem (a GRPO group). Samples are prompt-major: sample i answers problem i // samples_per_prompt
    start = time.time()
    samples = generate([chat_prompt_ids(problem["question"]) for problem in problems], samples_per_prompt)
    timing["gen"] = time.time() - start
    group_ids = [i // samples_per_prompt for i in range(len(samples))]

    # 2. Score each response, and 3. pad everything into one batch
    texts = [decode_tokens(sample["response_ids"]) for sample in samples]
    rewards = torch.tensor([reward_fn(text, problems[group]["answer"]) for text, group in zip(texts, group_ids)])
    batch = make_padded_batch([sample["prompt_ids"] for sample in samples], [sample["response_ids"] for sample in samples])

    # 4. Log-probs under the weights that generated the samples (π_old)
    start = time.time()
    old_log_probs = compute_log_probs(policy, batch)
    timing["old_log_prob"] = time.time() - start

    # 5. One advantage per response, copied onto each of its trained tokens
    advantages = grpo_advantages(rewards, group_ids)[:, None] * batch["response_mask"]
    batch.update(old_log_probs=old_log_probs, advantages=advantages)  # kept for inspection after the update

    # 6. PPO update
    start = time.time()
    stats = ppo_update(policy, optimizer, batch, old_log_probs, advantages, **update_kwargs)
    timing["update"] = time.time() - start

    metrics = {
        "reward": rewards.mean().item(),
        "format": float(np.mean([bool(BOXED.search(text)) for text in texts])),  # has a \boxed{integer} at all, right or wrong
        "response_len": batch["response_mask"].sum(-1).float().mean().item(),
        **stats,
    }
    if samples[0]["log_probs"] is not None:  # the inference engine returned its own log-probs: compare them with ours
        engine_log_probs = torch.zeros_like(old_log_probs)
        for row, sample in enumerate(samples):
            engine_log_probs[row, : len(sample["log_probs"])] = torch.tensor(sample["log_probs"])
        trained = batch["response_mask"].bool()
        metrics["logp_mismatch"] = (old_log_probs - engine_log_probs)[trained].abs().mean().item()
    metrics.update({f"time/{name}": seconds for name, seconds in timing.items()})
    return metrics, batch, samples

A small helper to print one line per step:

In [ ]:
def print_step_metrics(step: int, metrics: dict) -> None:
    fmt = f" | format {metrics['format']:.2f}" if "format" in metrics else ""
    mismatch = f" | logp mismatch {metrics['logp_mismatch']:.4f}" if "logp_mismatch" in metrics else ""
    times = " ".join(f"{name.removeprefix('time/')} {seconds:4.1f}s" for name, seconds in metrics.items() if name.startswith("time/"))
    print(f"step {step:2d} | reward {metrics['reward']:.3f}{fmt} | len {metrics['response_len']:5.1f} | grad_norm {metrics['grad_norm']:.3f}{mismatch} | {times}")

Now let’s create the optimizer and run 3 steps. Watch the timing — generation dominates, even for 64 short samples:

In [ ]:
LR = 2e-6
optimizer = make_optimizer(policy, LR)
step_rng = random.Random(SEED)
hf_history = []
for step in range(3):
    metrics, batch, samples = rl_step(policy, optimizer, lambda prompts, samples_per_prompt: hf_generate(policy, prompts, samples_per_prompt), step_rng.sample(TRAIN, 8), samples_per_prompt=8)
    hf_history.append(metrics)
    print_step_metrics(step, metrics)

That’s a complete RL trainer, in about 150 lines. Before we start replacing pieces of it, let’s make sure it actually does what we claimed.

### 1.7 Did it work?

The reward is far too noisy to judge a single step by, but we can check the update directly. `rl_step` kept the log-probs from *before* the update (`old_log_probs`) and the advantages of the last batch, so we can recompute the log-probs of exactly the same responses with the *updated* weights and look at two things:

1. **The surrogate objective went up.** Σ A·(ρ − 1) over trained tokens is the quantity a policy-gradient step is supposed to increase (to first order). Before the step, ρ = 1 everywhere, so it was exactly 0.
2. **Individual responses moved the intended way.** Responses with a positive advantage should have become more likely, and those with a negative advantage less likely.

In [ ]:
new_log_probs = compute_log_probs(policy, batch)  # under the updated weights
trained = batch["response_mask"].bool()
log_ratio = (new_log_probs - batch["old_log_probs"]) * trained  # log(π_new / π_old), per token
ratio_change = log_ratio.exp() - 1                               # how far each token's probability moved (0 = unchanged)
surrogate = (batch["advantages"] * ratio_change * trained).sum() / trained.sum()
print(f"surrogate objective, per trained token: 0 before the step -> {surrogate:+.2e} after")

Now let’s look at how individual responses shifted. Each dot is one response: its GRPO advantage on the x-axis, and how much its total log-probability changed on the y-axis. Points in the top-right and bottom-left quadrants moved the intended way.

In [ ]:
effect = pd.DataFrame({"advantage": batch["advantages"][:, 0].numpy(), "Δ log π(response)": log_ratio.sum(-1).numpy()})
for label, subset in [("A > 0", effect[effect.advantage > 0]), ("A < 0", effect[effect.advantage < 0])]:
    mean_change = subset["Δ log π(response)"].mean()
    moved_as_intended = (np.sign(subset["Δ log π(response)"]) == np.sign(subset.advantage)).mean()
    print(f"{label}: {len(subset):2d} responses, mean Δ log π(response) {mean_change:+.4f} "
          f"(probability × {np.exp(mean_change):.3f}), moved the intended way: {moved_as_intended:.0%}")
print(f"A = 0: {(effect.advantage == 0).sum()} responses (from groups where every attempt got the same reward)")

plt.figure(figsize=(6, 3.5))
plt.scatter(effect.advantage, effect["Δ log π(response)"], alpha=0.6)
plt.axhline(0, color="k", lw=0.5)
plt.axvline(0, color="k", lw=0.5)
plt.xlabel("GRPO advantage A")
plt.ylabel("Δ log π(response)")
plt.title(f"One optimizer step (lr = {LR:g})")
plt.tight_layout()
plt.show()

It worked. The surrogate objective went up, and a single optimizer step made the responses with a positive advantage about 1.4× more likely on average, and those with a negative advantage about 0.6× as likely. Often the bad responses drop further than the good ones rise: a correct response that was just sampled at temperature 1 often already has many tokens near log p = 0, so there’s little room left to push it up, while a wrong one can always be made less likely.

Not every response moved the intended way, though: most did, but around one in four or five didn’t. All responses share the same weights, so a step improves the objective as a whole rather than each response separately, which is one reason RL needs many samples and many small steps. The responses from same-reward groups had zero advantage, and contributed nothing at all.

Now let’s look at the timing printed by each step: **generation dominates**, even for 64 short samples. `model.generate` runs one Python-level step per token with no paging of the KV cache or continuous batching. At thousands of samples per step this is the bottleneck, and it’s the first thing every real framework replaces.

## 2. Swap 1: a real inference engine (vLLM)

§1.7 ended with the timing: generating the samples took far longer than learning from them. That's normal for RL on LLMs, and it's why every serious trainer hands generation to a dedicated **inference engine**. We'll use [vLLM](https://docs.vllm.ai).

**Why `model.generate` is slow.** It moves the whole batch forward together, one token per step, so three things go to waste:

* **Finished rows sit idle.** A response that stops after 50 tokens keeps its slot until the longest response in the batch finishes.
* **Memory is reserved up front.** Each row's key/value cache (the attention state for the tokens so far) is allocated as one contiguous block, big enough for the longest possible response, even if the row stops early. That caps how many rows fit on the GPU at once.
* **Python overhead every step.** Each new token launches hundreds of small GPU kernels from Python.

**What vLLM does instead:**

* **Continuous batching:** as soon as one sequence finishes, a waiting one takes its slot, so the GPU always works on a full batch.
* **PagedAttention:** the key/value cache is stored in small fixed-size blocks, handed out as a sequence grows, like pages of virtual memory. Nothing is reserved for tokens that never get written, so far more sequences fit at once.
* **CUDA graphs:** the GPU work for one decoding step is recorded once and replayed, skipping the per-step launch overhead.

```
static batching (model.generate)          continuous batching (vLLM)
slot 0:  A A A A A A A A                  slot 0:  A A A A A A A A
slot 1:  B B B · · · · ·  ← idle          slot 1:  B B B D D D D D  ← D starts as soon as B ends
slot 2:  C C C C C · · ·  ← idle          slot 2:  C C C C C E E E
```

We use vLLM's offline `LLM` class in this process (it starts its engine in a child process), on the same GPU as the policy, with 30% of GPU memory for its own copy of the weights and its key/value cache.

Two things change the moment generation leaves the training model:

1. **There are now two copies of the weights.** After every update, the trainer must push its new weights into vLLM. Otherwise the next samples come from an old version of the policy, and the training math (which assumes the samples came from the current policy) quietly stops being true.
2. **vLLM has its own sampling defaults**, and it takes them from the same `generation_config` we saw in §1.2.

In [ ]:
from vllm import LLM, SamplingParams, TokensPrompt

llm = LLM(model=MODEL, gpu_memory_utilization=0.3, max_model_len=4096, seed=SEED)
print("vLLM's default sampling params for this model:", llm.get_default_sampling_params())

These are the same chat settings we switched off in §1.2. If we called `llm.generate(prompts)` without our own `SamplingParams`, vLLM would silently sample from a top-k/top-p-truncated, repetition-penalised distribution: not the policy we're training. So `vllm_generate` always passes explicit, neutral parameters. verl does the same: it overrides the generation config when it launches vLLM.

`vllm_generate` also asks for `logprobs=0`. The number is how many *extra* top alternatives to return at each position, on top of the sampled token; 0 means "just the log-prob of the token you sampled", which is all we need (§1.4). Note that vLLM computes these from the *raw* model scores, before temperature scaling, so they only match the sampling distribution at temperature 1. RL rollouts use temperature 1 anyway.

What happens if we forget to override them? Let’s sample the same prompt twice — once with the model’s defaults (truncated distribution) and once with our RL settings (full distribution) — and compare the trainer’s log-probs of the results:

In [ ]:
# Sample with model defaults (the trap)
default_params = SamplingParams(n=8, max_tokens=128)
rl_params = SamplingParams(n=8, temperature=1.0, top_p=1.0, top_k=0, repetition_penalty=1.0, max_tokens=128, logprobs=0)

test_prompt = chat_prompt_ids(TRAIN[0]["question"])
default_out = llm.generate([TokensPrompt(prompt_token_ids=test_prompt)], default_params)
rl_out = llm.generate([TokensPrompt(prompt_token_ids=test_prompt)], rl_params)

def mean_token_log_prob(outputs):
    """Trainer's mean per-token log-prob of each sample (via a forward pass, not the engine's own log-probs)."""
    response_ids = [list(completion.token_ids) for request in outputs for completion in request.outputs]
    batch = make_padded_batch([test_prompt] * len(response_ids), response_ids)
    log_probs = compute_log_probs(policy, batch)
    mask = batch["response_mask"]
    return (log_probs * mask).sum(-1) / mask.sum(-1)

lp_default = mean_token_log_prob(default_out)
lp_rl = mean_token_log_prob(rl_out)
print(f"mean log π(response) with model defaults: {lp_default.mean():.2f}")
print(f"mean log π(response) with RL settings:    {lp_rl.mean():.2f}")
print(f"\ndefault samples are {torch.exp(lp_default.mean() - lp_rl.mean()):.1f}× more likely under the policy")

The default-sampled responses are several times more likely under the policy than the RL-sampled ones. That's the toy example from §1.2 in a real model: top-k and top-p threw away the unlikely tokens before sampling, so every response is made only of likely tokens. The trainer would see responses that look more typical of the policy than real samples are, and would never see the unlikely tokens at all, so it could never learn whether they were good or bad. It's a silent bug: training still runs, but learns about a distribution the model doesn't have. Always set the sampling parameters explicitly for RL.

**`vllm_generate`, step by step.** vLLM's output is nested, and the function just flattens it into the same sample dicts `hf_generate` produced:

```
request_outputs                      one per prompt
 └─ request_output.outputs           samples_per_prompt completions for that prompt
     ├─ completion.token_ids         the sampled response tokens
     └─ completion.logprobs          one dict per position: {sampled_token_id: Logprob(logprob=-0.03, ...)}
```

For each completion, we look up the sampled token in its position's dict to get its log-prob. Unlike `hf_generate`, we now get the engine's own log-probs for free, which is what lets `rl_step` report `logp_mismatch`. The cell also times both generators on the same 64 samples:

In [ ]:
def vllm_generate(llm, prompts: list[list[int]], samples_per_prompt: int, max_tokens: int = 256, temperature: float = 1.0) -> list[dict]:
    # logprobs=0: return the log-prob of each sampled token (and no alternatives)
    params = SamplingParams(n=samples_per_prompt, temperature=temperature, top_p=1.0, top_k=0, repetition_penalty=1.0, max_tokens=max_tokens, logprobs=0)
    token_prompts = [TokensPrompt(prompt_token_ids=prompt) for prompt in prompts]
    request_outputs = llm.generate(token_prompts, params, use_tqdm=False)
    samples = []
    for prompt, request_output in zip(prompts, request_outputs):
        for completion in request_output.outputs:  # samples_per_prompt completions per prompt
            response_ids = list(completion.token_ids)
            # completion.logprobs holds one {token_id: Logprob} dict per position; look up the token that was sampled
            log_probs = []
            for token_id, logprobs_at_position in zip(response_ids, completion.logprobs):
                log_probs.append(logprobs_at_position[token_id].logprob)
            samples.append({"prompt_ids": list(prompt), "response_ids": response_ids, "log_probs": log_probs})
    return samples


prompts = [chat_prompt_ids(problem["question"]) for problem in TRAIN[:8]]
generators = {"HF generate": lambda: hf_generate(policy, prompts, 8), "vLLM": lambda: vllm_generate(llm, prompts, 8)}
for name, generate in generators.items():
    start = time.time()
    generated = generate()
    print(f"{name:12} {len(generated)} samples in {time.time() - start:5.2f}s")
print("vLLM samples end with a stop token:", np.mean([sample["response_ids"][-1] in STOP_IDS for sample in generated]))

vLLM produced the same 64 samples about 10× faster, and the gap grows with batch size: a real RL step generates thousands of samples.

About 89% of the vLLM samples end with a stop token. The rest hit the 256-token limit (`max_tokens`) and were cut off mid-response. A cut-off response almost never contains a final `\boxed{}` answer, so it scores 0. That's a signal too: it teaches the model to finish within the budget.

### 2.1 Weight sync

The trainer updates its weights; vLLM needs those new weights before it generates the next batch. The simplest correct way takes two steps:

1. The trainer writes its weights to `/dev/shm` (a folder that lives in RAM, not on disk) in the Hugging Face checkpoint format, as bf16. vLLM computes in bf16 anyway, so it doesn't need the fp32 master weights. That's the **data plane**: the bulk of the bytes.
2. A tiny control message, `collective_rpc("reload_weights", ...)`, tells every vLLM worker to reload from that folder. That's the **control plane**: "new weights are ready, here's where".

Then we **reset vLLM's prefix cache**. vLLM remembers the key/value cache it computed for prompt beginnings it has already seen, and reuses it whenever a new prompt starts with the same tokens. Every prompt here begins with the same system prompt and chat-template tokens, so that's a big saving. But those cached values were computed with the *old* weights. After a weight update they're wrong, and vLLM would still reuse them, because the tokens match. Resetting the cache forces a recompute.

Two details in `save_for_inference`: Qwen2.5-0.5B **ties** its output layer to its input embedding (`tie_word_embeddings`, one matrix used for both), so we drop the duplicate `lm_head.weight` and vLLM ties them itself. And the reload prints a "RotaryEmbedding: Failed to load weights" warning. It's harmless: the rotary position embedding has no trained weights to load.

verl does the same thing faster. It sends bf16 tensors straight from the trainer's GPU memory to vLLM over CUDA IPC (same GPU) or NCCL (different GPUs), in buckets, with no copy through host memory. The structure is identical: a control call says *"new weights are coming"*, then a data channel moves them.

The policy already took 3 steps in §1, but vLLM loaded the original checkpoint from disk, so we sync once before doing anything else. Then we measure held-out accuracy with **greedy** decoding (temperature 0, always pick the most likely token). Training samples need randomness to explore; an accuracy measurement doesn't, and greedy makes it repeatable.

In [ ]:
import safetensors.torch


def save_for_inference(model, path: Path = SHM / "policy") -> str:
    path.mkdir(parents=True, exist_ok=True)
    state = {}
    for name, tensor in model.state_dict().items():
        state[name] = tensor.detach().to(torch.bfloat16).cpu().contiguous()  # a bf16 copy on the CPU, ready to write
    if model.config.tie_word_embeddings:
        state.pop("lm_head.weight", None)  # same tensor as the embedding; vLLM ties them itself
    safetensors.torch.save_file(state, str(path / "model.safetensors"))
    return str(path)


def push_weights(model, llm) -> None:
    weights_path = save_for_inference(model)                                     # 1. data plane: write the weights to RAM
    llm.collective_rpc("reload_weights", kwargs={"weights_path": weights_path})  # 2. control plane: tell vLLM to load them
    llm.reset_prefix_cache()  # cached KV values were computed with the old weights


t0 = time.time()
push_weights(policy, llm)
print(f"weights pushed in {time.time() - t0:.1f}s")


def evaluate(generate_greedy, problems=EVAL) -> float:
    """Accuracy with one greedy response per problem."""
    samples = generate_greedy([chat_prompt_ids(problem["question"]) for problem in problems], 1)
    rewards = [reward_fn(decode_tokens(sample["response_ids"]), problem["answer"]) for sample, problem in zip(samples, problems)]
    return float(np.mean(rewards))


acc_before = evaluate(lambda prompts, samples_per_prompt: vllm_generate(llm, prompts, samples_per_prompt, temperature=0.0))
print(f"greedy accuracy on {len(EVAL)} held-out problems before RL: {acc_before:.3f}")

### 2.2 Train, with vLLM generating

Same `rl_step` as before, now with `vllm_generate`, and with `push_weights` after every update. Each step samples 16 questions × 8 attempts = 128 rows; with `mini_batch_size=64`, that's two optimizer steps per training step.

How to read a step line:

| Field | Meaning | What to look for |
|---|---|---|
| `reward` | mean reward of this step's 128 samples | should trend up, noisily (16 questions is a small batch) |
| `format` | fraction of responses with a `\boxed{}` number at all, right or wrong | separates "learned the format" from "learned the arithmetic" |
| `len` | mean response length in tokens | how the model's behaviour is changing |
| `grad_norm` | size of the gradient before clipping (§1.5) | a sudden spike means an unusual batch |
| `logp mismatch` | mean \|log π_trainer − log π_vLLM\| over trained tokens | small and flat. vLLM and the trainer use different kernels, so below about 0.01 is normal. If we deleted the `push_weights` line it would creep up, as the trainer drifts away from what vLLM serves |
| `gen`, `old_log_prob`, `update`, `sync` | seconds spent in each phase | generation is now a small part of the step |

In [ ]:
vllm_history = []
for step in range(20):
    metrics, batch, samples = rl_step(policy, optimizer, lambda prompts, samples_per_prompt: vllm_generate(llm, prompts, samples_per_prompt), step_rng.sample(TRAIN, 16), samples_per_prompt=8)
    start = time.time()
    push_weights(policy, llm)
    metrics["time/sync"] = time.time() - start
    vllm_history.append(metrics)
    print_step_metrics(step, metrics)

acc_after = evaluate(lambda prompts, samples_per_prompt: vllm_generate(llm, prompts, samples_per_prompt, temperature=0.0))
print(f"\ngreedy accuracy on held-out problems: {acc_before:.3f} before -> {acc_after:.3f} after {len(vllm_history)} steps")

In [ ]:
def plot_histories(histories: dict[str, list[dict]], keys=("reward", "response_len")) -> None:
    fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3))
    for ax, key in zip(np.atleast_1d(axes), keys):
        for name, history in histories.items():
            ax.plot([metrics[key] for metrics in history], marker="o", ms=3, label=name)
        ax.set_title(key)
        ax.set_xlabel("step")
    np.atleast_1d(axes)[0].legend()
    plt.tight_layout()
    plt.show()


plot_histories({"§2 vLLM": vllm_history}, keys=("reward", "format", "response_len"))

Reward and format rise together at first: within about eight steps almost every response ends with a `\boxed{}`, up from under 90%. A response without one always scores 0, so that alone is worth over ten points of reward. After that, the format has nowhere left to go, but the reward and the greedy held-out accuracy (0.83 → 0.945) keep improving, noisily, so the model is getting more of the arithmetic right too. Responses also got shorter, from about 140 tokens to about 115.

This is the "literal" point from §1.1 in action. The first thing RL teaches is usually whatever the reward checks most directly, so a big early jump in reward is rarely a jump in ability; it's worth tracking the format separately, as we did here.

### 2.3 The trainer and vLLM don't quite agree

`logp_mismatch` is an average, and averages hide things. In the last batch, let's look at the distribution of the per-token difference between the trainer's log-probs (`old_log_probs`, recomputed by our policy) and the log-probs vLLM reported when it sampled the same tokens. In PPO terms, exp(difference) is the ratio ρ that the very first update would see if we used vLLM's numbers as π_old: it should be exactly 1.

In [ ]:
trained = batch["response_mask"].bool()
engine_log_probs = torch.zeros_like(batch["old_log_probs"])
for row, sample in enumerate(samples):
    engine_log_probs[row, : len(sample["log_probs"])] = torch.tensor(sample["log_probs"])
diff = (batch["old_log_probs"] - engine_log_probs)[trained]  # log π_trainer − log π_vLLM, one entry per trained token
print(f"{diff.numel()} trained tokens: mean |Δ| = {diff.abs().mean():.4f}, 99th percentile |Δ| = {diff.abs().quantile(0.99):.3f}, max |Δ| = {diff.abs().max():.3f}")
print(f"implied ratio exp(Δ) ranges over [{diff.exp().min():.3f}, {diff.exp().max():.3f}], before any update")

plt.figure(figsize=(6, 3))
plt.hist(diff.numpy(), bins=100, log=True)
plt.xlabel("log π_trainer − log π_vLLM, per token")
plt.ylabel("tokens (log scale)")
plt.tight_layout()
plt.show()

Most tokens agree almost exactly, so the mean difference is tiny, but the distribution has a long tail: one token in a hundred is off by about 0.1 or more, and the worst by about 0.4.

Why does that matter? Suppose we had used vLLM's log-probs as π_old in the PPO ratio. Take a token where the trainer says log π = −2.00 and vLLM said −2.26. Before any update, the ratio would be e^(−2.00 − (−2.26)) = e^0.26 ≈ 1.30. That's already past the clip limit of 1.28 from §1.5, so if the token had a positive advantage, its gradient would be zero: the clip would block an update that never happened, purely because two programs compute the same number slightly differently.

That's why `rl_step` recomputes `old_log_probs` with the trainer instead of trusting vLLM's numbers: the PPO ratio must compare like with like, so that ρ = 1 exactly before the first update. verl does the same (its `compute_log_prob` step), keeps vLLM's numbers as `rollout_log_probs`, and can use the gap for importance-sampling corrections (`algorithm.rollout_correction`). At scale, this train/inference mismatch is one of the main sources of RL instability, which is why verl can log it every step (`rollout_corr/kl`, which we'll see in §9.2).

## What we built, and what comes next

| We wrote | It became | Why |
|---|---|---|
| `hf_generate` | vLLM | generation speed: paged KV cache, continuous batching, CUDA graphs |
| (nothing: one copy of the weights) | `save_for_inference` + `reload_weights` | two copies of the weights must be kept in sync after every update |

We now have a complete RL trainer that learns, with a real inference engine. Every function we wrote is also in `tutorial/lib/rl_core.py`, copied verbatim, so the next notebooks can import it instead of re-running this one.

**Next:** [notebook 2](2_from_trainer_to_verl.ipynb) rebuilds this trainer from verl's pieces, one swap at a time, until it *is* verl. First, free the GPU:

In [ ]:
llm.llm_engine.engine_core.shutdown()
del llm, policy, optimizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {gpu_free_gib():.0f} GiB")